In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, r2_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, LogisticRegression

# 1. Generate Realistic Synthetic Dataset
np.random.seed(42)
n = 1200

age = np.random.randint(18, 70, size=n)
gender = np.random.choice(['Male', 'Female'], size=n)
income = np.random.randint(20000, 150000, size=n)
spending = np.random.randint(100, 10000, size=n)
freq = np.random.randint(1, 50, size=n)
aov = np.random.uniform(10.0, 500.0, size=n).round(2)
recency = np.random.randint(1, 365, size=n)
visits = np.random.randint(1, 100, size=n)
discount = np.random.uniform(0.0, 0.8, size=n).round(2)

# Generate dependent targets with clear underlying relationships
rating = (2.0 + 0.0002 * spending + 0.03 * freq - 0.003 * recency + np.random.normal(0, 0.4, n)).clip(1.0, 5.0).round(1)
prob = 1 / (1 + np.exp(-(0.05 * freq - 0.01 * recency + 0.5 * rating - 2.5)))
will_purchase = (prob > 0.5).astype(int)

df = pd.DataFrame({
    'CustomerID': [f'CUST_{1000 + i}' for i in range(n)],
    'Age': age,
    'Gender': gender,
    'AnnualIncome': income,
    'TotalSpending': spending,
    'PurchaseFrequency': freq,
    'AverageOrderValue': aov,
    'DaysSinceLastPurchase': recency,
    'WebsiteVisits': visits,
    'DiscountUsage': discount,
    'CustomerRating': rating,
    'WillPurchaseAgain': will_purchase
})

# Save customer_data.csv
df.to_csv('customer_data.csv', index=False)
print("✅ Saved 'customer_data.csv' successfully!")

# 2. Preprocessing & Outlier Handling (IQR Capping)
num_cols = ['AnnualIncome', 'TotalSpending', 'PurchaseFrequency', 'AverageOrderValue', 'DaysSinceLastPurchase']
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = np.clip(df[col], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

# 3. K-Means Clustering (k=5)
features = ['DaysSinceLastPurchase', 'PurchaseFrequency', 'TotalSpending', 'DiscountUsage']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

segment_map = {
    0: 'High-Value Loyal',
    1: 'At-Risk',
    2: 'Discount-Driven',
    3: 'New & Promising',
    4: 'Low-Engagement'
}
df['Segment_Name'] = df['Cluster'].map(segment_map)

# Save customer_segments.csv
df[['CustomerID', 'Cluster', 'Segment_Name']].to_csv('customer_segments.csv', index=False)
print("✅ Saved 'customer_segments.csv' successfully!")

# 4. Ridge Regression (CustomerRating)
X_reg = df[['AnnualIncome', 'TotalSpending', 'PurchaseFrequency', 'AverageOrderValue', 'DaysSinceLastPurchase', 'DiscountUsage']]
y_reg = df['CustomerRating']
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

ridge = GridSearchCV(Ridge(), {'alpha': [0.1, 1.0, 10.0, 100.0]}, cv=5, scoring='r2')
ridge.fit(X_tr_r, y_tr_r)
print(f"\n📊 Ridge Regression R² Score: {r2_score(y_te_r, ridge.predict(X_te_r)):.2f}")

# 5. Logistic Regression (WillPurchaseAgain)
X_cls = df[['Age', 'AnnualIncome', 'TotalSpending', 'PurchaseFrequency', 'DaysSinceLastPurchase', 'CustomerRating', 'DiscountUsage']]
y_cls = df['WillPurchaseAgain']
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

log_reg = GridSearchCV(LogisticRegression(max_iter=1000), {'C': [0.01, 0.1, 1, 10]}, cv=5, scoring='f1')
log_reg.fit(X_tr_c, y_tr_c)
print("\n📊 Classification Performance:")
print(classification_report(y_te_c, log_reg.predict(X_te_c)))

✅ Saved 'customer_data.csv' successfully!
✅ Saved 'customer_segments.csv' successfully!

📊 Ridge Regression R² Score: 0.77


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c


📊 Classification Performance:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       191
           1       1.00      0.96      0.98        49

    accuracy                           0.99       240
   macro avg       0.99      0.98      0.99       240
weighted avg       0.99      0.99      0.99       240

